# Phase 1B D11-D13 Acceptance Notebook

Human-readable acceptance checks for walk-forward integrity, the post-hoc DSR heuristic screen, and additional baseline strategies.


## 0. Setup

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
import os
import json
import sqlite3
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display



def _bootstrap_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "backtest").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            os.chdir(candidate)
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


PROJECT_ROOT = _bootstrap_repo_root()
print("Notebook repo root:", PROJECT_ROOT)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 160)

DB_PATH = PROJECT_ROOT / "backtest" / "experiments.db"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
PARQUET_PATH = PROJECT_ROOT / "data" / "raw" / "btcusdt_1h.parquet"
ENV_PATH = PROJECT_ROOT / "config" / "environments.yaml"

assert PARQUET_PATH.exists(), f"Missing canonical parquet: {PARQUET_PATH}"
assert ENV_PATH.exists(), f"Missing environments.yaml: {ENV_PATH}"

D11_ACCEPTANCE = {}


def pass_fail(condition):
    return "PASS" if bool(condition) else "FAIL"


def display_check_table(rows, title=None, require_all=True):
    checks = pd.DataFrame(rows)
    if title:
        print(title)
    display(checks)
    if require_all:
        failed = checks[checks["status"] != "PASS"]
        assert failed.empty, failed
    return checks


def check_row(check, passed, detail=""):
    return {"check": check, "status": pass_fail(passed), "detail": detail}


def read_runs_query(query, params=None):
    assert DB_PATH.exists(), f"experiments.db not found: {DB_PATH}"
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(query, conn, params=params or ())


## A. Run Real Walk-Forward

This is the real D11 acceptance run. It uses the CLI path so the behavior matches the user-facing command:

```bash
python -m backtest.engine --strategy sma_crossover --mode walk-forward
```

The command writes a `walk_forward_summary` row plus one `walk_forward_window` row per generated window into `backtest/experiments.db`.

In [2]:
RUN_WALK_FORWARD = True

walk_forward_command = [
    sys.executable,
    "-m",
    "backtest.engine",
    "--strategy",
    "sma_crossover",
    "--mode",
    "walk-forward",
]

if RUN_WALK_FORWARD:
    completed = subprocess.run(walk_forward_command, capture_output=True, text=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    assert completed.returncode == 0, "Walk-forward CLI run failed"
else:
    print("Walk-forward run skipped. Set RUN_WALK_FORWARD = True for D11 acceptance.")

D11_ACCEPTANCE["walk_forward_cli_run"] = RUN_WALK_FORWARD



WALK-FORWARD SUMMARY
Summary ID:   92eb97ae...
Strategy:     sma_crossover
Windows:      20
Mean Return:    0.3748 (37.48%)
Mean Sharpe:    -0.178
Worst Drawdown: 0.4366 (43.66%)
Total Trades:   513
Win Rate:       0.32
Profit Factor:  1.07

Window   Train Period             Test Period               Trades   Return   Sharpe
-------- ------------------------ ------------------------ ------- -------- --------
  1      2020-01-01 to 2020-12-31 2021-01-01 to 2021-03-31     128   2.7109    2.240
  2      2020-04-01 to 2021-03-31 2021-04-01 to 2021-06-30     123   1.5435    1.656
  3      2020-07-01 to 2021-06-30 2021-07-01 to 2021-09-30     123   0.9169    1.230
  4      2020-10-01 to 2021-09-30 2021-10-01 to 2021-12-31     121   1.1132    1.357
  5      2021-01-01 to 2021-12-31 2022-01-01 to 2022-03-31     121   0.0103    0.282
  6      2021-04-01 to 2022-03-31 2022-04-01 to 2022-06-30     127  -0.5748   -1.265
  7      2021-07-01 to 2022-06-30 2022-07-01 to 2022-09-30     128  -0.3488  

## B. Fetch Latest Summary Row

The latest `walk_forward_summary` row is the parent for this acceptance run.

In [3]:
summary_df = read_runs_query("""
SELECT *
FROM runs
WHERE run_type = 'walk_forward_summary'
ORDER BY created_at_utc DESC
LIMIT 1
""")

display(summary_df.T)

assert len(summary_df) == 1, "Expected exactly one latest walk_forward_summary row"
summary = summary_df.iloc[0]
summary_run_id = summary["run_id"]
print("summary_run_id:", summary_run_id)

summary_checks = [
    check_row("summary row exists", len(summary_df) == 1),
    check_row("summary run_type", summary["run_type"] == "walk_forward_summary", summary["run_type"]),
    check_row("summary parent_run_id is NULL", pd.isna(summary["parent_run_id"]), summary["parent_run_id"]),
    check_row("summary strategy_name", summary["strategy_name"] == "sma_crossover", summary["strategy_name"]),
    check_row("summary fee_model", summary["fee_model"] == "effective_7bps_per_side", summary["fee_model"]),
]
display_check_table(summary_checks, "Summary row checks")
D11_ACCEPTANCE["summary_row"] = True


,0
run_id,92eb97ae-08e6-482d-8c1d-85313ed258f3
run_type,walk_forward_summary
parent_run_id,None
strategy_name,sma_crossover
strategy_source,manual
git_commit,8411d22
config_hash,sha256:57d14658de08d92d
data_snapshot_date,None
feature_version,none
split_version,v1


summary_run_id: 92eb97ae-08e6-482d-8c1d-85313ed258f3
Summary row checks


,check,status,detail
0,summary row exists,PASS,
1,summary run_type,PASS,walk_forward_summary
2,summary parent_run_id is NULL,PASS,NaN
3,summary strategy_name,PASS,sma_crossover
4,summary fee_model,PASS,effective_7bps_per_side


## C. Fetch Window Rows

Each window row must have `run_type = walk_forward_window` and `parent_run_id = summary_run_id`.

In [4]:
windows_df = read_runs_query("""
SELECT *
FROM runs
WHERE run_type = 'walk_forward_window'
  AND parent_run_id = ?
ORDER BY test_start
""", params=(summary_run_id,))

window_display_cols = [
    "run_id",
    "parent_run_id",
    "train_start",
    "train_end",
    "test_start",
    "test_end",
    "initial_capital",
    "final_capital",
    "total_trades",
    "sharpe_ratio",
    "total_return",
    "max_drawdown",
]
display(windows_df[window_display_cols])

assert len(windows_df) > 0, "No walk_forward_window rows found"
D11_ACCEPTANCE["window_rows_loaded"] = True
print(f"Loaded {len(windows_df)} window rows for summary {summary_run_id}")


,run_id,parent_run_id,train_start,train_end,test_start,test_end,initial_capital,final_capital,total_trades,sharpe_ratio,total_return,max_drawdown
0,9d2c8087-a445-46a3-b6a0-d4b123c471f0,92eb97ae-08e6-482d-8c1d-85313ed258f3,2020-01-01T00:00:00Z,2020-12-31T23:00:00Z,2021-01-01T00:00:00Z,2021-03-31T23:00:00Z,10000.0,37109.271310,20,2.461450,2.710927,0.199996
1,b6ef1f53-4dc2-4dab-8bef-3ee13c48de0d,92eb97ae-08e6-482d-8c1d-85313ed258f3,2020-04-01T00:00:00Z,2021-03-31T23:00:00Z,2021-04-01T00:00:00Z,2021-06-30T23:00:00Z,10000.0,25435.072353,24,-2.115818,1.543507,0.386292
2,8df34f98-d3b0-4baf-8df7-0ef951c098d8,92eb97ae-08e6-482d-8c1d-85313ed258f3,2020-07-01T00:00:00Z,2021-06-30T23:00:00Z,2021-07-01T00:00:00Z,2021-09-30T23:00:00Z,10000.0,19168.502049,25,0.081637,0.916850,0.205315
3,764eba1f-f5d2-4d23-abc5-fa897fa36643,92eb97ae-08e6-482d-8c1d-85313ed258f3,2020-10-01T00:00:00Z,2021-09-30T23:00:00Z,2021-10-01T00:00:00Z,2021-12-31T23:00:00Z,10000.0,21132.322255,26,0.277892,1.113232,0.317930
4,f2d94cb7-56bf-40cf-b623-ecdd2d7bded0,92eb97ae-08e6-482d-8c1d-85313ed258f3,2021-01-01T00:00:00Z,2021-12-31T23:00:00Z,2022-01-01T00:00:00Z,2022-03-31T23:00:00Z,10000.0,10102.785476,23,0.291954,0.010279,0.179252
5,d5437466-eeb7-40dd-9dc9-411b1bc78811,92eb97ae-08e6-482d-8c1d-85313ed258f3,2021-04-01T00:00:00Z,2022-03-31T23:00:00Z,2022-04-01T00:00:00Z,2022-06-30T23:00:00Z,10000.0,4251.980238,27,-5.491045,-0.574802,0.436579
6,e18f6460-ddbe-44b1-a6e4-7ebb621fadc7,92eb97ae-08e6-482d-8c1d-85313ed258f3,2021-07-01T00:00:00Z,2022-06-30T23:00:00Z,2022-07-01T00:00:00Z,2022-09-30T23:00:00Z,10000.0,6512.450965,25,1.095467,-0.348755,0.185761
7,708abf83-9e86-40f4-8e99-da94fd27bf78,92eb97ae-08e6-482d-8c1d-85313ed258f3,2021-10-01T00:00:00Z,2022-09-30T23:00:00Z,2022-10-01T00:00:00Z,2022-12-31T23:00:00Z,10000.0,5575.784864,29,-1.227828,-0.442422,0.108024
8,72ecc637-828c-4617-a430-58748f7a885f,92eb97ae-08e6-482d-8c1d-85313ed258f3,2022-01-01T00:00:00Z,2022-12-31T23:00:00Z,2023-01-01T00:00:00Z,2023-03-31T23:00:00Z,10000.0,9502.994118,25,4.328595,-0.049701,0.111195
9,9b9a0898-5dc3-482b-93f5-b2319a19eb94,92eb97ae-08e6-482d-8c1d-85313ed258f3,2022-04-01T00:00:00Z,2023-03-31T23:00:00Z,2023-04-01T00:00:00Z,2023-06-30T23:00:00Z,10000.0,9191.967108,29,0.029617,-0.080803,0.157481


Loaded 20 window rows for summary 92eb97ae-08e6-482d-8c1d-85313ed258f3


## D. Window Generation Matches `environments.yaml`

Compare registry rows against `generate_walk_forward_windows()` using the config values from `config/environments.yaml`.

In [5]:
from backtest.engine import generate_walk_forward_windows

env_config = yaml.safe_load(ENV_PATH.read_text())
wf_config = env_config["walk_forward"]
train_months = int(wf_config["train_window_months"])
test_months = int(wf_config["test_window_months"])
step_months = int(wf_config["step_months"])
overall_start = date.fromisoformat(env_config["splits"]["training"]["start"])
overall_end = date.fromisoformat(env_config["splits"]["test"]["end"])

expected_windows = generate_walk_forward_windows(
    overall_start=overall_start,
    overall_end=overall_end,
    train_months=train_months,
    test_months=test_months,
    step_months=step_months,
)

expected_windows_df = pd.DataFrame(
    expected_windows,
    columns=["expected_train_start", "expected_train_end", "expected_test_start", "expected_test_end"],
)
actual_windows_df = windows_df[["train_start", "train_end", "test_start", "test_end"]].copy()
for col in actual_windows_df.columns:
    actual_windows_df[col] = pd.to_datetime(actual_windows_df[col], utc=True).dt.date

comparison_df = pd.concat([expected_windows_df, actual_windows_df], axis=1)
display(comparison_df)

window_generation_checks = [
    check_row("window count matches generated tuples", len(windows_df) == len(expected_windows), f"actual={len(windows_df)}, expected={len(expected_windows)}"),
    check_row("train_start matches", actual_windows_df["train_start"].tolist() == expected_windows_df["expected_train_start"].tolist()),
    check_row("train_end matches", actual_windows_df["train_end"].tolist() == expected_windows_df["expected_train_end"].tolist()),
    check_row("test_start matches", actual_windows_df["test_start"].tolist() == expected_windows_df["expected_test_start"].tolist()),
    check_row("test_end matches", actual_windows_df["test_end"].tolist() == expected_windows_df["expected_test_end"].tolist()),
]
display_check_table(window_generation_checks, "Window generation checks")
D11_ACCEPTANCE["window_generation"] = True


,expected_train_start,expected_train_end,expected_test_start,expected_test_end,train_start,train_end,test_start,test_end
0,2020-01-01,2020-12-31,2021-01-01,2021-03-31,2020-01-01,2020-12-31,2021-01-01,2021-03-31
1,2020-04-01,2021-03-31,2021-04-01,2021-06-30,2020-04-01,2021-03-31,2021-04-01,2021-06-30
2,2020-07-01,2021-06-30,2021-07-01,2021-09-30,2020-07-01,2021-06-30,2021-07-01,2021-09-30
3,2020-10-01,2021-09-30,2021-10-01,2021-12-31,2020-10-01,2021-09-30,2021-10-01,2021-12-31
4,2021-01-01,2021-12-31,2022-01-01,2022-03-31,2021-01-01,2021-12-31,2022-01-01,2022-03-31
5,2021-04-01,2022-03-31,2022-04-01,2022-06-30,2021-04-01,2022-03-31,2022-04-01,2022-06-30
6,2021-07-01,2022-06-30,2022-07-01,2022-09-30,2021-07-01,2022-06-30,2022-07-01,2022-09-30
7,2021-10-01,2022-09-30,2022-10-01,2022-12-31,2021-10-01,2022-09-30,2022-10-01,2022-12-31
8,2022-01-01,2022-12-31,2023-01-01,2023-03-31,2022-01-01,2022-12-31,2023-01-01,2023-03-31
9,2022-04-01,2023-03-31,2023-04-01,2023-06-30,2022-04-01,2023-03-31,2023-04-01,2023-06-30


Window generation checks


,check,status,detail
0,window count matches generated tuples,PASS,"actual=20, expected=20"
1,train_start matches,PASS,
2,train_end matches,PASS,
3,test_start matches,PASS,
4,test_end matches,PASS,


## E. Four-Item Isolation Checklist

These are the core D11 isolation checks:

1. Every window has a distinct `run_id`.
2. Every window points to the same parent summary.
3. Every window resets `initial_capital` instead of inheriting previous `final_capital`.
4. Every window's trade artifact contains only trades inside its test period.

In [6]:
# Checklist 1: unique run_id
unique_run_id = windows_df["run_id"].nunique() == len(windows_df)

# Checklist 2: same parent summary
one_parent = windows_df["parent_run_id"].nunique() == 1
parent_matches = bool(windows_df["parent_run_id"].iloc[0] == summary_run_id)

# Checklist 3: capital reset
initial_capital_unique = windows_df["initial_capital"].nunique() == 1
expected_initial = float(windows_df["initial_capital"].iloc[0])
initial_is_baseline = np.isclose(expected_initial, 10_000.0, rtol=0, atol=1e-9)
prev_finals = windows_df["final_capital"].shift(1)
capital_leak_mask = np.isclose(windows_df["initial_capital"], prev_finals, rtol=0, atol=1e-9)
capital_leak_mask = pd.Series(capital_leak_mask, index=windows_df.index).fillna(False)
no_capital_inheritance = not bool(capital_leak_mask.any())

capital_table = windows_df[["run_id", "initial_capital", "final_capital"]].copy()
capital_table["prev_final_capital"] = prev_finals
capital_table["looks_like_inherited_capital"] = capital_leak_mask
display(capital_table)

isolation_checks = [
    check_row("each window has unique run_id", unique_run_id, f"unique={windows_df['run_id'].nunique()}, rows={len(windows_df)}"),
    check_row("all windows share one parent_run_id", one_parent, str(windows_df["parent_run_id"].unique())),
    check_row("parent_run_id matches summary_run_id", parent_matches, summary_run_id),
    check_row("initial_capital identical across windows", initial_capital_unique, str(windows_df["initial_capital"].unique())),
    check_row("initial_capital resets to 10000", initial_is_baseline, expected_initial),
    check_row("no later window initial equals prior final", no_capital_inheritance, capital_table.loc[capital_table["looks_like_inherited_capital"], ["run_id", "initial_capital", "prev_final_capital"]].to_dict(orient="records")),
]
display_check_table(isolation_checks, "Window identity/parent/capital isolation checks")
D11_ACCEPTANCE["identity_parent_capital_isolation"] = True


,run_id,initial_capital,final_capital,prev_final_capital,looks_like_inherited_capital
0,9d2c8087-a445-46a3-b6a0-d4b123c471f0,10000.0,37109.271310,NaN,False
1,b6ef1f53-4dc2-4dab-8bef-3ee13c48de0d,10000.0,25435.072353,37109.271310,False
2,8df34f98-d3b0-4baf-8df7-0ef951c098d8,10000.0,19168.502049,25435.072353,False
3,764eba1f-f5d2-4d23-abc5-fa897fa36643,10000.0,21132.322255,19168.502049,False
4,f2d94cb7-56bf-40cf-b623-ecdd2d7bded0,10000.0,10102.785476,21132.322255,False
5,d5437466-eeb7-40dd-9dc9-411b1bc78811,10000.0,4251.980238,10102.785476,False
6,e18f6460-ddbe-44b1-a6e4-7ebb621fadc7,10000.0,6512.450965,4251.980238,False
7,708abf83-9e86-40f4-8e99-da94fd27bf78,10000.0,5575.784864,6512.450965,False
8,72ecc637-828c-4617-a430-58748f7a885f,10000.0,9502.994118,5575.784864,False
9,9b9a0898-5dc3-482b-93f5-b2319a19eb94,10000.0,9191.967108,9502.994118,False


Window identity/parent/capital isolation checks


,check,status,detail
0,each window has unique run_id,PASS,"unique=20, rows=20"
1,all windows share one parent_run_id,PASS,"<ArrowStringArray>\n['92eb97ae-08e6-482d-8c1d-85313ed258f3']\nLength: 1, dtype: str"
2,parent_run_id matches summary_run_id,PASS,92eb97ae-08e6-482d-8c1d-85313ed258f3
3,initial_capital identical across windows,PASS,[10000.]
4,initial_capital resets to 10000,PASS,10000.0
5,no later window initial equals prior final,PASS,[]


### E4. Trade Artifact Window Isolation

This is the strongest check for no trade/position leakage. For every window row, load `data/results/trades_{run_id}.csv` and verify all completed trade entry/exit times fall within that window's `test_start` to `test_end` range.

If this fails, D11 should not be signed off: it means the persisted artifact is not test-window isolated even if registry metrics are.

In [7]:
trade_artifact_rows = []
trade_leak_rows = []

for _, row in windows_df.iterrows():
    run_id = row["run_id"]
    trade_path = RESULTS_DIR / f"trades_{run_id}.csv"
    test_start = pd.Timestamp(row["test_start"], tz="UTC") if pd.Timestamp(row["test_start"]).tzinfo is None else pd.Timestamp(row["test_start"])
    test_end = pd.Timestamp(row["test_end"], tz="UTC") if pd.Timestamp(row["test_end"]).tzinfo is None else pd.Timestamp(row["test_end"])

    artifact = {
        "run_id": run_id,
        "trade_path": str(trade_path),
        "trade_csv_exists": trade_path.exists(),
        "registry_total_trades": int(row["total_trades"]),
        "csv_rows": None,
        "all_entries_in_test": None,
        "all_exits_in_test": None,
        "row_count_matches_registry": None,
    }

    if not trade_path.exists():
        trade_leak_rows.append({"run_id": run_id, "issue": "missing trade CSV", "detail": str(trade_path)})
        trade_artifact_rows.append(artifact)
        continue

    trades = pd.read_csv(
        trade_path,
        parse_dates=["entry_time_utc", "exit_time_utc", "entry_signal_time_utc", "exit_signal_time_utc"],
    )
    artifact["csv_rows"] = len(trades)
    artifact["row_count_matches_registry"] = len(trades) == int(row["total_trades"])

    if len(trades) == 0:
        artifact["all_entries_in_test"] = True
        artifact["all_exits_in_test"] = True
    else:
        entry_ok = (trades["entry_time_utc"] >= test_start) & (trades["entry_time_utc"] <= test_end)
        exit_ok = (trades["exit_time_utc"] >= test_start) & (trades["exit_time_utc"] <= test_end)
        artifact["all_entries_in_test"] = bool(entry_ok.all())
        artifact["all_exits_in_test"] = bool(exit_ok.all())

        leaked = trades[~(entry_ok & exit_ok)].copy()
        if len(leaked) > 0:
            leaked.insert(0, "run_id", run_id)
            leaked.insert(1, "test_start", test_start)
            leaked.insert(2, "test_end", test_end)
            trade_leak_rows.extend(leaked[[
                "run_id", "test_start", "test_end", "trade_id",
                "entry_signal_time_utc", "entry_time_utc",
                "exit_signal_time_utc", "exit_time_utc",
            ]].to_dict(orient="records"))

    trade_artifact_rows.append(artifact)

trade_artifact_df = pd.DataFrame(trade_artifact_rows)
display(trade_artifact_df)

if trade_leak_rows:
    leak_df = pd.DataFrame(trade_leak_rows)
    display(leak_df.head(30))
else:
    leak_df = pd.DataFrame()

trade_isolation_checks = [
    check_row("all per-window trade CSVs exist", bool(trade_artifact_df["trade_csv_exists"].all())),
    check_row("CSV row counts match registry total_trades", bool(trade_artifact_df["row_count_matches_registry"].all()), trade_artifact_df.loc[~trade_artifact_df["row_count_matches_registry"].fillna(False)].to_dict(orient="records")),
    check_row("all trade entries are inside their test window", bool(trade_artifact_df["all_entries_in_test"].all()), f"leaked_rows={len(leak_df)}"),
    check_row("all trade exits are inside their test window", bool(trade_artifact_df["all_exits_in_test"].all()), f"leaked_rows={len(leak_df)}"),
]
display_check_table(trade_isolation_checks, "Trade artifact isolation checks")
D11_ACCEPTANCE["trade_artifact_isolation"] = True


,run_id,trade_path,trade_csv_exists,registry_total_trades,csv_rows,all_entries_in_test,all_exits_in_test,row_count_matches_registry
0,9d2c8087-a445-46a3-b6a0-d4b123c471f0,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_9d2c8087-a445-46a3-b6a0-d4b123c471f0.csv,True,20,20,True,True,True
1,b6ef1f53-4dc2-4dab-8bef-3ee13c48de0d,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_b6ef1f53-4dc2-4dab-8bef-3ee13c48de0d.csv,True,24,24,True,True,True
2,8df34f98-d3b0-4baf-8df7-0ef951c098d8,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_8df34f98-d3b0-4baf-8df7-0ef951c098d8.csv,True,25,25,True,True,True
3,764eba1f-f5d2-4d23-abc5-fa897fa36643,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_764eba1f-f5d2-4d23-abc5-fa897fa36643.csv,True,26,26,True,True,True
4,f2d94cb7-56bf-40cf-b623-ecdd2d7bded0,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_f2d94cb7-56bf-40cf-b623-ecdd2d7bded0.csv,True,23,23,True,True,True
5,d5437466-eeb7-40dd-9dc9-411b1bc78811,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_d5437466-eeb7-40dd-9dc9-411b1bc78811.csv,True,27,27,True,True,True
6,e18f6460-ddbe-44b1-a6e4-7ebb621fadc7,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_e18f6460-ddbe-44b1-a6e4-7ebb621fadc7.csv,True,25,25,True,True,True
7,708abf83-9e86-40f4-8e99-da94fd27bf78,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_708abf83-9e86-40f4-8e99-da94fd27bf78.csv,True,29,29,True,True,True
8,72ecc637-828c-4617-a430-58748f7a885f,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_72ecc637-828c-4617-a430-58748f7a885f.csv,True,25,25,True,True,True
9,9b9a0898-5dc3-482b-93f5-b2319a19eb94,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_9b9a0898-5dc3-482b-93f5-b2319a19eb94.csv,True,29,29,True,True,True


Trade artifact isolation checks


,check,status,detail
0,all per-window trade CSVs exist,PASS,
1,CSV row counts match registry total_trades,PASS,[]
2,all trade entries are inside their test window,PASS,leaked_rows=0
3,all trade exits are inside their test window,PASS,leaked_rows=0


## F. Summary Aggregation Consistency

The summary row must be mechanically reproducible from the window rows:

- mean Sharpe
- mean return
- summed trades
- worst drawdown
- max drawdown duration
- notes JSON with `num_windows` and config values

In [8]:
notes = json.loads(summary["notes"]) if isinstance(summary["notes"], str) and summary["notes"] else {}
print("Summary notes:")
display(pd.DataFrame([notes]).T.rename(columns={0: "value"}))

calc_mean_sharpe = float(windows_df["sharpe_ratio"].mean())
calc_mean_return = float(windows_df["total_return"].mean())
calc_total_trades = int(windows_df["total_trades"].sum())
calc_worst_drawdown = float(windows_df["max_drawdown"].max())
calc_max_dd_duration = float(windows_df["max_drawdown_duration_hours"].max())
calc_num_windows = len(windows_df)

summary_calc_df = pd.DataFrame([
    {"metric": "mean_sharpe", "calculated": calc_mean_sharpe, "summary_row": summary["sharpe_ratio"]},
    {"metric": "mean_return", "calculated": calc_mean_return, "summary_row": summary["total_return"]},
    {"metric": "total_trades", "calculated": calc_total_trades, "summary_row": summary["total_trades"]},
    {"metric": "worst_drawdown", "calculated": calc_worst_drawdown, "summary_row": summary["max_drawdown"]},
    {"metric": "max_drawdown_duration", "calculated": calc_max_dd_duration, "summary_row": summary["max_drawdown_duration_hours"]},
    {"metric": "num_windows", "calculated": calc_num_windows, "summary_row": notes.get("num_windows")},
])
display(summary_calc_df)

summary_agg_checks = [
    check_row("summary sharpe is mean window sharpe", np.isclose(summary["sharpe_ratio"], calc_mean_sharpe, rtol=1e-10, atol=1e-10)),
    check_row("summary return is mean window return", np.isclose(summary["total_return"], calc_mean_return, rtol=1e-10, atol=1e-10)),
    check_row("summary total_trades is sum windows", int(summary["total_trades"]) == calc_total_trades),
    check_row("summary max_drawdown is worst window drawdown", np.isclose(summary["max_drawdown"], calc_worst_drawdown, rtol=1e-10, atol=1e-10)),
    check_row("summary max drawdown duration is max window duration", np.isclose(summary["max_drawdown_duration_hours"], calc_max_dd_duration, rtol=1e-10, atol=1e-10)),
    check_row("notes num_windows matches", notes.get("num_windows") == calc_num_windows, notes.get("num_windows")),
    check_row("notes train_months matches config", notes.get("train_months") == train_months, notes.get("train_months")),
    check_row("notes test_months matches config", notes.get("test_months") == test_months, notes.get("test_months")),
    check_row("notes step_months matches config", notes.get("step_months") == step_months, notes.get("step_months")),
]
display_check_table(summary_agg_checks, "Summary aggregation checks")
D11_ACCEPTANCE["summary_aggregation"] = True


Summary notes:


,value
num_windows,20
train_months,12
test_months,3
step_months,3
params,{}


,metric,calculated,summary_row
0,mean_sharpe,-0.178381,-0.178381
1,mean_return,0.374825,0.374825
2,total_trades,513.000000,513.000000
3,worst_drawdown,0.436579,0.436579
4,max_drawdown_duration,2145.000000,2145.000000
5,num_windows,20.000000,20.000000


Summary aggregation checks


,check,status,detail
0,summary sharpe is mean window sharpe,PASS,
1,summary return is mean window return,PASS,
2,summary total_trades is sum windows,PASS,
3,summary max_drawdown is worst window drawdown,PASS,
4,summary max drawdown duration is max window duration,PASS,
5,notes num_windows matches,PASS,20
6,notes train_months matches config,PASS,12
7,notes test_months matches config,PASS,3
8,notes step_months matches config,PASS,3


## G. Single-Run Backward Compatibility

D11 touched registry writing and engine orchestration, so single-run mode must still work unchanged.

In [11]:
from backtest.engine import run_backtest
from strategies.baseline.sma_crossover import SMACrossover

single_result = run_backtest(
    strategy_cls=SMACrossover,
    start_date=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2024, 6, 30, 23, tzinfo=timezone.utc),
    parquet_path=PARQUET_PATH,
    cash=10_000.0,
    write_registry=True,
)

single_df = read_runs_query("""
SELECT *
FROM runs
WHERE run_id = ?
""", params=(single_result.run_id,))
assert len(single_df) == 1, "Single-run registry row missing"
single_row = single_df.iloc[0]

display(single_df[[
    "run_id", "run_type", "parent_run_id", "strategy_name",
    "test_start", "test_end", "initial_capital", "final_capital",
    "total_trades", "sharpe_ratio", "total_return"
]])

single_trade_csv = RESULTS_DIR / f"trades_{single_result.run_id}.csv"
single_trades_df = pd.read_csv(single_trade_csv) if single_trade_csv.exists() else pd.DataFrame()

single_artifact_summary = pd.DataFrame([
    {"source": "result.trades", "trade_rows": len(single_result.trades)},
    {"source": "registry total_trades", "trade_rows": int(single_row["total_trades"])},
    {"source": "trade CSV rows", "trade_rows": len(single_trades_df)},
])
display(single_artifact_summary)

single_checks = [
    check_row("single run_type remains single_run", single_row["run_type"] == "single_run", single_row["run_type"]),
    check_row("single parent_run_id is NULL", pd.isna(single_row["parent_run_id"]), single_row["parent_run_id"]),
    check_row("single registry strategy is sma_crossover", single_row["strategy_name"] == "sma_crossover", single_row["strategy_name"]),
    check_row("single trade CSV exists", single_trade_csv.exists(), str(single_trade_csv)),
    check_row("single result has trades", single_result.metrics["total_trades"] > 0, single_result.metrics["total_trades"]),
    check_row(
        "single trade CSV row count equals result.trades",
        len(single_trades_df) == len(single_result.trades),
        f"csv_rows={len(single_trades_df)}, result_trades={len(single_result.trades)}",
    ),
    check_row(
        "single trade CSV row count equals registry total_trades",
        len(single_trades_df) == int(single_row["total_trades"]),
        f"csv_rows={len(single_trades_df)}, registry_total_trades={single_row['total_trades']}",
    ),
]
display_check_table(single_checks, "Single-run backward compatibility checks")
D11_ACCEPTANCE["single_run_backward_compatibility"] = True


2026-04-16T05:48:49Z [INFO] Starting backtest: strategy=sma_crossover, range=[2024-01-01, 2024-06-30], run_id=89732640
2026-04-16T05:48:49Z [INFO] Loading parquet feed: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/raw/btcusdt_1h.parquet
2026-04-16T05:48:49Z [INFO]   Loaded 4368 bars: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
2026-04-16T05:48:49Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T05:48:49Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%
2026-04-16T05:48:50Z [INFO] Metrics: return=-0.0067 sharpe=0.141 maxDD=0.3178 trades=56 win_rate=0.38 PF=0.96
2026-04-16T05:48:50Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_89732640-5c6e-4671-8e42-df806119910a.csv (56 trades)
2026-04-16T05:48:50Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T05:48:50Z [INFO] Inserted run 89732640-5c6e-4671-8e42-df806

,run_id,run_type,parent_run_id,strategy_name,test_start,test_end,initial_capital,final_capital,total_trades,sharpe_ratio,total_return
0,89732640-5c6e-4671-8e42-df806119910a,single_run,None,sma_crossover,2024-01-01T00:00:00Z,2024-06-30T23:00:00Z,10000.0,9932.736819,56,0.141497,-0.006726


,source,trade_rows
0,result.trades,56
1,registry total_trades,56
2,trade CSV rows,56


Single-run backward compatibility checks


,check,status,detail
0,single run_type remains single_run,PASS,single_run
1,single parent_run_id is NULL,PASS,None
2,single registry strategy is sma_crossover,PASS,sma_crossover
3,single trade CSV exists,PASS,/Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_89732640-5c6e-4671-8e42-df806119910a.csv
4,single result has trades,PASS,56
5,single trade CSV row count equals result.trades,PASS,"csv_rows=56, result_trades=56"
6,single trade CSV row count equals registry total_trades,PASS,"csv_rows=56, registry_total_trades=56"


## H. D11 Final Acceptance Summary

In [10]:
d11_rows = [
    {"criterion": "walk-forward CLI completed", "status": pass_fail(D11_ACCEPTANCE.get("walk_forward_cli_run")), "evidence": "python -m backtest.engine --strategy sma_crossover --mode walk-forward"},
    {"criterion": "summary row exists", "status": pass_fail(D11_ACCEPTANCE.get("summary_row")), "evidence": "latest walk_forward_summary row fetched"},
    {"criterion": "window rows loaded", "status": pass_fail(D11_ACCEPTANCE.get("window_rows_loaded")), "evidence": "walk_forward_window rows with parent_run_id"},
    {"criterion": "window tuples match environments.yaml", "status": pass_fail(D11_ACCEPTANCE.get("window_generation")), "evidence": "registry dates match generate_walk_forward_windows()"},
    {"criterion": "run_id / parent / capital isolation", "status": pass_fail(D11_ACCEPTANCE.get("identity_parent_capital_isolation")), "evidence": "unique run_id, common summary parent, capital reset"},
    {"criterion": "trade artifact isolation", "status": pass_fail(D11_ACCEPTANCE.get("trade_artifact_isolation")), "evidence": "per-window trade CSV rows constrained to test windows"},
    {"criterion": "summary aggregation", "status": pass_fail(D11_ACCEPTANCE.get("summary_aggregation")), "evidence": "summary metrics recomputed from window rows"},
    {"criterion": "single-run backward compatibility", "status": pass_fail(D11_ACCEPTANCE.get("single_run_backward_compatibility")), "evidence": "single_run registry row and trade CSV still work"},
]

d11_summary_df = pd.DataFrame(d11_rows)
display(d11_summary_df)

failed = d11_summary_df[d11_summary_df["status"] != "PASS"]
assert failed.empty, failed

print("D11 WALK-FORWARD ACCEPTANCE PASS")
print("summary_run_id:", summary_run_id)
print("windows:", len(windows_df))


,criterion,status,evidence
0,walk-forward CLI completed,PASS,python -m backtest.engine --strategy sma_crossover --mode walk-forward
1,summary row exists,PASS,latest walk_forward_summary row fetched
2,window rows loaded,PASS,walk_forward_window rows with parent_run_id
3,window tuples match environments.yaml,PASS,registry dates match generate_walk_forward_windows()
4,run_id / parent / capital isolation,PASS,"unique run_id, common summary parent, capital reset"
5,trade artifact isolation,PASS,per-window trade CSV rows constrained to test windows
6,summary aggregation,PASS,summary metrics recomputed from window rows
7,single-run backward compatibility,PASS,single_run registry row and trade CSV still work


D11 WALK-FORWARD ACCEPTANCE PASS
summary_run_id: 92eb97ae-08e6-482d-8c1d-85313ed258f3
windows: 20


## I. D12 Post-Hoc DSR Heuristic Acceptance

D12 is accepted as a heuristic multiple-testing screen, not as a production-grade or formal Deflated Sharpe Ratio certification. These cells verify formula math, decision logic, real registry query filters, and report language.


In [16]:
from math import isclose, log, sqrt

from backtest.evaluate_dsr import (
    compute_expected_max_sharpe,
    evaluate_trials,
    query_sharpe_ratios,
)

D12_SPLIT_VERSION = "v1"
D12_DEFAULT_RUN_TYPE = "walk_forward_summary"
D12_ACCEPTANCE = {}


## J. Threshold Math

Checks the expected-max-Sharpe heuristic formula directly: `sqrt(2 * ln(N))`, with `N <= 1` returning `0`.


In [17]:
threshold_ns = [0, 1, 2, 5, 20, 100]
threshold_rows = []

for n in threshold_ns:
    actual = compute_expected_max_sharpe(n)
    expected = 0.0 if n <= 1 else sqrt(2 * log(n))
    threshold_rows.append({
        "N": n,
        "actual_threshold": actual,
        "expected_threshold": expected,
        "matches_formula": isclose(actual, expected, rel_tol=0, abs_tol=1e-12),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

positive_thresholds = threshold_df[threshold_df["N"] >= 1]["actual_threshold"].to_numpy()
threshold_checks = [
    check_row("N <= 1 returns zero", (threshold_df.loc[threshold_df["N"] <= 1, "actual_threshold"] == 0.0).all()),
    check_row("threshold equals sqrt(2 * ln(N))", threshold_df["matches_formula"].all()),
    check_row("threshold is monotonic non-decreasing", np.all(np.diff(positive_thresholds) >= 0)),
]
display_check_table(threshold_checks, "D12 threshold math checks")
D12_ACCEPTANCE["threshold_math"] = True


,N,actual_threshold,expected_threshold,matches_formula
0,0,0.000000,0.000000,True
1,1,0.000000,0.000000,True
2,2,1.177410,1.177410,True
3,5,1.794123,1.794123,True
4,20,2.447747,2.447747,True
5,100,3.034854,3.034854,True


D12 threshold math checks


,check,status,detail
0,N <= 1 returns zero,PASS,
1,threshold equals sqrt(2 * ln(N)),PASS,
2,threshold is monotonic non-decreasing,PASS,


## K. Decision Logic

Uses synthetic Sharpe trials so the acceptance boundary is visible: only `best_sharpe > threshold` survives; equality does not survive; empty input must not crash.


In [18]:
equal_threshold = compute_expected_max_sharpe(5)
decision_cases = [
    {
        "name": "N=1 positive Sharpe survives",
        "data": {"a": 0.5},
        "expected_survives": True,
    },
    {
        "name": "N=20 best=0.5 fails",
        "data": {f"s{i}": (0.5 if i == 0 else 0.1) for i in range(20)},
        "expected_survives": False,
    },
    {
        "name": "N=5 best=3.0 survives",
        "data": {"a": 3.0, "b": 0.2, "c": -0.1, "d": 0.5, "e": 0.3},
        "expected_survives": True,
    },
    {
        "name": "N=5 best equals threshold fails",
        "data": {"a": equal_threshold, "b": 0.2, "c": -0.1, "d": 0.5, "e": 0.3},
        "expected_survives": False,
    },
    {
        "name": "empty input does not crash",
        "data": {},
        "expected_survives": False,
    },
]

decision_rows = []
for case in decision_cases:
    result = evaluate_trials(case["data"])
    decision_rows.append({
        "case": case["name"],
        "N": result["n_trials"],
        "best_sharpe": result["best_sharpe"],
        "threshold": result["threshold"],
        "expected_survives": case["expected_survives"],
        "actual_survives": result["survives"],
        "pass": result["survives"] == case["expected_survives"],
        "rankings": result["rankings"],
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

decision_checks = [
    check_row("synthetic decision cases match expected boundary", decision_df["pass"].all(), decision_df.loc[~decision_df["pass"], ["case", "rankings"]].to_dict(orient="records")),
]
display_check_table(decision_checks, "D12 decision logic checks")
D12_ACCEPTANCE["decision_logic"] = True


,case,N,best_sharpe,threshold,expected_survives,actual_survives,pass,rankings
0,N=1 positive Sharpe survives,1,0.500000,0.000000,True,True,True,"[{'strategy': 'a', 'sharpe': 0.5}]"
1,N=20 best=0.5 fails,20,0.500000,2.447747,False,False,True,"[{'strategy': 's0', 'sharpe': 0.5}, {'strategy': 's1', 'sharpe': 0.1}, {'strategy': 's2', 'sharpe': 0.1}, {'strategy': 's3', 'sharpe': 0.1}, {'strategy': 's..."
2,N=5 best=3.0 survives,5,3.000000,1.794123,True,True,True,"[{'strategy': 'a', 'sharpe': 3.0}, {'strategy': 'd', 'sharpe': 0.5}, {'strategy': 'e', 'sharpe': 0.3}, {'strategy': 'b', 'sharpe': 0.2}, {'strategy': 'c', '..."
3,N=5 best equals threshold fails,5,1.794123,1.794123,False,False,True,"[{'strategy': 'a', 'sharpe': 1.7941225779941015}, {'strategy': 'd', 'sharpe': 0.5}, {'strategy': 'e', 'sharpe': 0.3}, {'strategy': 'b', 'sharpe': 0.2}, {'st..."
4,empty input does not crash,0,NaN,0.000000,False,False,True,[]


D12 decision logic checks


,check,status,detail
0,synthetic decision cases match expected boundary,PASS,[]


## L. Registry Query Filters

Queries real `experiments.db` rows and cross-checks `query_sharpe_ratios()` against direct SQLite counts. This validates split, run type, strategy, and NULL-Sharpe exclusion semantics.


In [19]:
rows_default = query_sharpe_ratios(split_version=D12_SPLIT_VERSION)
rows_single = query_sharpe_ratios(split_version=D12_SPLIT_VERSION, run_type="single_run")
rows_sma = query_sharpe_ratios(split_version=D12_SPLIT_VERSION, strategy_name="sma_crossover")

with sqlite3.connect(DB_PATH) as conn:
    db_default = pd.read_sql_query("""
        SELECT run_id, strategy_name, run_type, split_version, sharpe_ratio
        FROM runs
        WHERE split_version = ?
          AND run_type = ?
          AND sharpe_ratio IS NOT NULL
        ORDER BY sharpe_ratio DESC
    """, conn, params=(D12_SPLIT_VERSION, D12_DEFAULT_RUN_TYPE))
    db_single = pd.read_sql_query("""
        SELECT run_id, strategy_name, run_type, split_version, sharpe_ratio
        FROM runs
        WHERE split_version = ?
          AND run_type = 'single_run'
          AND sharpe_ratio IS NOT NULL
        ORDER BY sharpe_ratio DESC
    """, conn, params=(D12_SPLIT_VERSION,))
    db_sma = pd.read_sql_query("""
        SELECT run_id, strategy_name, run_type, split_version, sharpe_ratio
        FROM runs
        WHERE split_version = ?
          AND run_type = ?
          AND strategy_name = 'sma_crossover'
          AND sharpe_ratio IS NOT NULL
        ORDER BY sharpe_ratio DESC
    """, conn, params=(D12_SPLIT_VERSION, D12_DEFAULT_RUN_TYPE))
    db_all_split = pd.read_sql_query("""
        SELECT run_id, strategy_name, run_type, split_version, sharpe_ratio
        FROM runs
        WHERE split_version = ?
        ORDER BY created_at_utc
    """, conn, params=(D12_SPLIT_VERSION,))

query_summary = pd.DataFrame([
    {"query": "default walk_forward_summary", "query_rows": len(rows_default), "db_non_null_rows": len(db_default)},
    {"query": "single_run", "query_rows": len(rows_single), "db_non_null_rows": len(db_single)},
    {"query": "sma_crossover walk_forward_summary", "query_rows": len(rows_sma), "db_non_null_rows": len(db_sma)},
])
display(query_summary)
display(db_all_split.head())
print("NULL Sharpe rows in split:", int(db_all_split["sharpe_ratio"].isna().sum()))
print("First query rows:", list(rows_default.items())[:5] if rows_default else "no rows")

query_checks = [
    check_row("default query returns real registry data", len(rows_default) > 0, f"split_version={D12_SPLIT_VERSION}, run_type={D12_DEFAULT_RUN_TYPE}"),
    check_row("split_version + default run_type count matches DB", len(rows_default) == len(db_default), query_summary.to_dict(orient="records")),
    check_row("run_type filter count matches DB", len(rows_single) == len(db_single), query_summary.to_dict(orient="records")),
    check_row("strategy_name filter count matches DB", len(rows_sma) == len(db_sma), query_summary.to_dict(orient="records")),
    check_row("query excludes NULL Sharpe", all(pd.notna(value) for value in [*rows_default.values(), *rows_single.values(), *rows_sma.values()])),
]
display_check_table(query_checks, "D12 registry query checks")
D12_ACCEPTANCE["registry_query_filters"] = True
D12_ACCEPTANCE["null_sharpe_exclusion"] = True


2026-04-16T06:08:37Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T06:08:37Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T06:08:37Z [INFO] Ensured 'runs' table exists with Phase 1A schema


,query,query_rows,db_non_null_rows
0,default walk_forward_summary,2,2
1,single_run,9,9
2,sma_crossover walk_forward_summary,2,2


,run_id,strategy_name,run_type,split_version,sharpe_ratio
0,6445c6a0-0b73-49ce-a517-fcd2ed045987,sma_crossover,single_run,v1,0.000000
1,7112415b-9c0b-47c1-815c-ce45004e8154,sma_crossover,single_run,v1,0.000000
2,14af73e6-6033-4fd8-8f50-25d0d5c0bcf5,sma_crossover,single_run,v1,1.046638
3,29d531c0-a7e3-40fe-bf68-ad245ba8e98b,sma_crossover,single_run,v1,1.046638
4,a45204b5-0c9b-4f5f-9bd1-61662cf671e1,sma_crossover,single_run,v1,1.046638


NULL Sharpe rows in split: 0
First query rows: [('sma_crossover (04d89e60)', -0.17838148214377175), ('sma_crossover (92eb97ae)', -0.17838148214377175)]
D12 registry query checks


,check,status,detail
0,default query returns real registry data,PASS,"split_version=v1, run_type=walk_forward_summary"
1,split_version + default run_type count matches DB,PASS,"[{'query': 'default walk_forward_summary', 'query_rows': 2, 'db_non_null_rows': 2}, {'query': 'single_run', 'query_rows': 9, 'db_non_null_rows': 9}, {'query..."
2,run_type filter count matches DB,PASS,"[{'query': 'default walk_forward_summary', 'query_rows': 2, 'db_non_null_rows': 2}, {'query': 'single_run', 'query_rows': 9, 'db_non_null_rows': 9}, {'query..."
3,strategy_name filter count matches DB,PASS,"[{'query': 'default walk_forward_summary', 'query_rows': 2, 'db_non_null_rows': 2}, {'query': 'single_run', 'query_rows': 9, 'db_non_null_rows': 9}, {'query..."
4,query excludes NULL Sharpe,PASS,


## M. CLI Report Semantics

Runs the real CLI and keeps the text output visible in the notebook. The key acceptance point is that the report labels itself as an approximate / heuristic screen rather than a formal DSR certification.


In [20]:
def run_dsr_cli(*args):
    cmd = [sys.executable, "-m", "backtest.evaluate_dsr", *args]
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    output = (completed.stdout or "") + ("\n" + completed.stderr if completed.stderr else "")
    print("$", " ".join(cmd))
    print(output)
    assert completed.returncode == 0, f"DSR CLI failed with exit code {completed.returncode}"
    return output

cli_outputs = {
    "default_walk_forward_summary": run_dsr_cli("--split-version", D12_SPLIT_VERSION),
    "single_run": run_dsr_cli("--split-version", D12_SPLIT_VERSION, "--run-type", "single_run"),
    "sma_crossover": run_dsr_cli("--split-version", D12_SPLIT_VERSION, "--strategy", "sma_crossover"),
}

combined_output = "\n".join(cli_outputs.values()).lower()
semantic_checks = [
    check_row("report includes number of trials", "trials" in combined_output),
    check_row("report includes best/worst/mean Sharpe", all(term in combined_output for term in ["best sharpe", "worst sharpe", "mean sharpe"])),
    check_row("report includes threshold", "threshold" in combined_output),
    check_row("report includes survive/not-survive decision", "surviv" in combined_output),
    check_row("report includes strategy ranking", "rank" in combined_output),
    check_row("report explicitly says heuristic and approximate", ("heuristic" in combined_output) and ("approximate" in combined_output)),
    check_row("report says it is not full DSR", ("not the" in combined_output) and ("deflated sharpe ratio" in combined_output)),
]
display_check_table(semantic_checks, "D12 CLI report semantics checks")
D12_ACCEPTANCE["cli_report_output"] = True
D12_ACCEPTANCE["explicit_heuristic_disclaimer"] = True


$ /Users/yutianyang/miniconda3/bin/python -m backtest.evaluate_dsr --split-version v1

  MULTIPLE-TESTING EVALUATION (Heuristic Approximation)

  NOTE: This is an approximate screen using the expected
  maximum Sharpe threshold SR* = sqrt(2 * ln(N)), NOT the
  full Bailey-López de Prado Deflated Sharpe Ratio.

  Trials (N):           2
  Threshold SR*:        1.1774
  Best Sharpe:          -0.1784
  Worst Sharpe:         -0.1784
  Mean Sharpe:          -0.1784

  Best strategy DOES NOT SURVIVE multiple-testing correction
  (-0.1784 <= 1.1774)

  Rank     Sharpe  Strategy
  ------ --------  ----------------------------------------
  1       -0.1784  sma_crossover (04d89e60)
  2       -0.1784  sma_crossover (92eb97ae)



2026-04-16T06:08:37Z [INFO] Ensured 'runs' table exists with Phase 1A schema

$ /Users/yutianyang/miniconda3/bin/python -m backtest.evaluate_dsr --split-version v1 --run-type single_run

  MULTIPLE-TESTING EVALUATION (Heuristic Approximation)

  NOTE: This is an approxim

,check,status,detail
0,report includes number of trials,PASS,
1,report includes best/worst/mean Sharpe,PASS,
2,report includes threshold,PASS,
3,report includes survive/not-survive decision,PASS,
4,report includes strategy ranking,PASS,
5,report explicitly says heuristic and approximate,PASS,
6,report says it is not full DSR,PASS,


## N. D12 Final Acceptance Summary

D12 passes when formula math, decision logic, registry query filters, NULL-Sharpe exclusion, CLI output, and heuristic disclaimer checks all pass.


In [21]:
d12_rows = [
    check_row("threshold math", D12_ACCEPTANCE.get("threshold_math", False)),
    check_row("decision logic", D12_ACCEPTANCE.get("decision_logic", False)),
    check_row("registry query filters", D12_ACCEPTANCE.get("registry_query_filters", False)),
    check_row("NULL Sharpe exclusion", D12_ACCEPTANCE.get("null_sharpe_exclusion", False)),
    check_row("CLI report output", D12_ACCEPTANCE.get("cli_report_output", False)),
    check_row("explicit heuristic disclaimer", D12_ACCEPTANCE.get("explicit_heuristic_disclaimer", False)),
]
display_check_table(d12_rows, "D12 final acceptance summary")
print("D12 ACCEPTED as a heuristic multiple-testing screen, not a production-grade formal DSR implementation.")


D12 final acceptance summary


,check,status,detail
0,threshold math,PASS,
1,decision logic,PASS,
2,registry query filters,PASS,
3,NULL Sharpe exclusion,PASS,
4,CLI report output,PASS,
5,explicit heuristic disclaimer,PASS,


D12 ACCEPTED as a heuristic multiple-testing screen, not a production-grade formal DSR implementation.


## O. D13 Additional Baselines Acceptance

D13 acceptance checks that `volatility_breakout` and `mean_reversion` behave like normal Phase 1B strategies: discoverable by the engine, runnable in single-run and walk-forward modes, isolated in registry/artifacts, and sane relative to the existing baselines.


In [22]:
from backtest.engine import (
    STRATEGY_REGISTRY,
    _ensure_strategies_loaded,
    run_backtest,
    run_walk_forward,
)
from strategies.baseline.mean_reversion import MeanReversion
from strategies.baseline.momentum import Momentum
from strategies.baseline.sma_crossover import SMACrossover
from strategies.baseline.volatility_breakout import VolatilityBreakout

D13_ACCEPTANCE = {}
D13_SINGLE_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
D13_SINGLE_END = datetime(2024, 6, 30, 23, tzinfo=timezone.utc)
D13_CASH = 10_000.0
D13_FEE_RATE = 0.0007

D13_NEW_STRATEGIES = {
    "volatility_breakout": VolatilityBreakout,
    "mean_reversion": MeanReversion,
}
D13_ALL_BASELINES = {
    "sma_crossover": SMACrossover,
    "momentum": Momentum,
    **D13_NEW_STRATEGIES,
}


def finite_number(value):
    return value is not None and np.isfinite(float(value))


def as_utc_timestamp(value):
    ts = pd.Timestamp(value)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def load_trade_csv(run_id):
    path = RESULTS_DIR / f"trades_{run_id}.csv"
    if not path.exists():
        return path, pd.DataFrame()
    trades = pd.read_csv(
        path,
        parse_dates=[
            "entry_signal_time_utc", "entry_time_utc",
            "exit_signal_time_utc", "exit_time_utc",
        ],
    )
    for col in [
        "entry_signal_time_utc", "entry_time_utc",
        "exit_signal_time_utc", "exit_time_utc",
    ]:
        trades[col] = pd.to_datetime(trades[col], utc=True)
    return path, trades


## P. Strategy Discovery

Confirms that engine auto-discovery sees all four baseline strategies, including the two D13 additions.


In [23]:
_ensure_strategies_loaded()
strategy_registry_df = pd.DataFrame([
    {"strategy_name": name, "class": cls.__name__, "module": cls.__module__}
    for name, cls in sorted(STRATEGY_REGISTRY.items())
])
display(strategy_registry_df)

expected_discovery = set(D13_ALL_BASELINES)
discovery_checks = [
    check_row("all four baseline strategies discovered", expected_discovery <= set(STRATEGY_REGISTRY)),
    check_row("volatility_breakout maps to VolatilityBreakout", STRATEGY_REGISTRY.get("volatility_breakout") is VolatilityBreakout),
    check_row("mean_reversion maps to MeanReversion", STRATEGY_REGISTRY.get("mean_reversion") is MeanReversion),
]
display_check_table(discovery_checks, "D13 strategy discovery checks")
D13_ACCEPTANCE["discovery"] = True


,strategy_name,class,module
0,mean_reversion,MeanReversion,strategies.baseline.mean_reversion
1,momentum,Momentum,strategies.baseline.momentum
2,sma_crossover,SMACrossover,strategies.baseline.sma_crossover
3,volatility_breakout,VolatilityBreakout,strategies.baseline.volatility_breakout


D13 strategy discovery checks


,check,status,detail
0,all four baseline strategies discovered,PASS,
1,volatility_breakout maps to VolatilityBreakout,PASS,
2,mean_reversion maps to MeanReversion,PASS,


## Q. Single-Run Spot Checks

Runs each D13 strategy on a fixed 2024 H1 range and checks that registry rows, trade artifacts, and finite metrics are produced.


In [24]:
D13_SINGLE_RESULTS = {}
single_rows = []

for strategy_name, strategy_cls in D13_NEW_STRATEGIES.items():
    result = run_backtest(
        strategy_cls=strategy_cls,
        start_date=D13_SINGLE_START,
        end_date=D13_SINGLE_END,
        parquet_path=PARQUET_PATH,
        cash=D13_CASH,
        write_registry=True,
    )
    D13_SINGLE_RESULTS[strategy_name] = result

    registry = read_runs_query("""
        SELECT *
        FROM runs
        WHERE run_id = ?
    """, params=(result.run_id,))
    trade_path, trades = load_trade_csv(result.run_id)

    single_rows.append({
        "strategy_name": strategy_name,
        "run_id": result.run_id,
        "warmup_bars": result.warmup_bars,
        "result_trades": len(result.trades),
        "registry_rows": len(registry),
        "registry_total_trades": int(registry.iloc[0]["total_trades"]) if len(registry) else None,
        "trade_csv_exists": trade_path.exists(),
        "trade_csv_rows": len(trades),
        "sharpe_ratio": result.metrics["sharpe_ratio"],
        "total_return": result.metrics["total_return"],
        "max_drawdown": result.metrics["max_drawdown"],
    })

single_run_df = pd.DataFrame(single_rows)
display(single_run_df)

single_checks = [
    check_row("both D13 strategies completed", set(single_run_df["strategy_name"]) == set(D13_NEW_STRATEGIES)),
    check_row("each single-run has registry row", (single_run_df["registry_rows"] == 1).all()),
    check_row("each single-run produces trades", (single_run_df["result_trades"] > 0).all(), single_run_df.to_dict(orient="records")),
    check_row("trade CSV exists for each single-run", single_run_df["trade_csv_exists"].all()),
    check_row("CSV rows match result trades", (single_run_df["trade_csv_rows"] == single_run_df["result_trades"]).all()),
    check_row("registry total_trades matches result trades", (single_run_df["registry_total_trades"] == single_run_df["result_trades"]).all()),
    check_row("single-run metrics are finite", single_run_df[["sharpe_ratio", "total_return", "max_drawdown"]].map(finite_number).all().all()),
    check_row("new strategy Sharpe is not suspiciously high", (single_run_df["sharpe_ratio"] < 2.0).all(), single_run_df[["strategy_name", "sharpe_ratio"]].to_dict(orient="records")),
]
display_check_table(single_checks, "D13 single-run checks")
D13_ACCEPTANCE["single_run"] = True


2026-04-16T06:37:02Z [INFO] Starting backtest: strategy=volatility_breakout, range=[2024-01-01, 2024-06-30], run_id=4f51640e
2026-04-16T06:37:02Z [INFO] Loading parquet feed: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/raw/btcusdt_1h.parquet
2026-04-16T06:37:02Z [INFO]   Loaded 4368 bars: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
2026-04-16T06:37:02Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T06:37:02Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%
2026-04-16T06:37:03Z [INFO] Metrics: return=0.1010 sharpe=0.826 maxDD=0.1474 trades=80 win_rate=0.36 PF=1.12
2026-04-16T06:37:03Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_4f51640e-3cde-4f23-beb2-2a4cc2770cef.csv (80 trades)
2026-04-16T06:37:03Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T06:37:03Z [INFO] Inserted run 4f51640e-3cde-4f23-beb2-

,strategy_name,run_id,warmup_bars,result_trades,registry_rows,registry_total_trades,trade_csv_exists,trade_csv_rows,sharpe_ratio,total_return,max_drawdown
0,volatility_breakout,4f51640e-3cde-4f23-beb2-2a4cc2770cef,24,80,1,80,True,80,0.825532,0.101031,0.147404
1,mean_reversion,893c9858-b3b5-4f41-b63c-f61513091f96,48,46,1,46,True,46,1.365857,0.203350,0.153280


D13 single-run checks


,check,status,detail
0,both D13 strategies completed,PASS,
1,each single-run has registry row,PASS,
2,each single-run produces trades,PASS,"[{'strategy_name': 'volatility_breakout', 'run_id': '4f51640e-3cde-4f23-beb2-2a4cc2770cef', 'warmup_bars': 24, 'result_trades': 80, 'registry_rows': 1, 'reg..."
3,trade CSV exists for each single-run,PASS,
4,CSV rows match result trades,PASS,
5,registry total_trades matches result trades,PASS,
6,single-run metrics are finite,PASS,
7,new strategy Sharpe is not suspiciously high,PASS,"[{'strategy_name': 'volatility_breakout', 'sharpe_ratio': 0.8255320000440055}, {'strategy_name': 'mean_reversion', 'sharpe_ratio': 1.3658570447389082}]"


## R. Walk-Forward Window Coverage

Runs walk-forward for each D13 strategy and prints per-window test bars, warmup, trade count, Sharpe, and return. The key check is that no test window is effectively empty or swallowed by warmup.


In [25]:
D13_WF_RESULTS = {}
D13_WINDOW_ROWS = []
raw_for_d13 = pd.read_parquet(PARQUET_PATH)
raw_for_d13["open_time_utc"] = pd.to_datetime(raw_for_d13["open_time_utc"], utc=True)

for strategy_name, strategy_cls in D13_NEW_STRATEGIES.items():
    wf_result = run_walk_forward(
        strategy_cls=strategy_cls,
        parquet_path=PARQUET_PATH,
        cash=D13_CASH,
    )
    D13_WF_RESULTS[strategy_name] = wf_result

    summary_df_d13 = read_runs_query("""
        SELECT *
        FROM runs
        WHERE run_id = ?
    """, params=(wf_result.summary_run_id,))
    assert len(summary_df_d13) == 1, f"Missing D13 summary row for {strategy_name}"

    window_df_d13 = read_runs_query("""
        SELECT *
        FROM runs
        WHERE run_type = 'walk_forward_window'
          AND parent_run_id = ?
        ORDER BY test_start
    """, params=(wf_result.summary_run_id,))

    for _, row in window_df_d13.iterrows():
        test_start = as_utc_timestamp(row["test_start"])
        test_end = as_utc_timestamp(row["test_end"])
        num_test_bars = int(((raw_for_d13["open_time_utc"] >= test_start) & (raw_for_d13["open_time_utc"] <= test_end)).sum())
        D13_WINDOW_ROWS.append({
            "strategy_name": strategy_name,
            "summary_run_id": wf_result.summary_run_id,
            "run_id": row["run_id"],
            "test_start": row["test_start"],
            "test_end": row["test_end"],
            "num_test_bars": num_test_bars,
            "warmup_bars": int(row["warmup_bars"]),
            "total_trades": int(row["total_trades"]),
            "sharpe_ratio": row["sharpe_ratio"],
            "total_return": row["total_return"],
            "max_drawdown": row["max_drawdown"],
        })

D13_WINDOW_DF = pd.DataFrame(D13_WINDOW_ROWS)
display(D13_WINDOW_DF)

zero_trade_summary = D13_WINDOW_DF.groupby("strategy_name")["total_trades"].agg(
    windows="count",
    zero_trade_windows=lambda s: int((s == 0).sum()),
    total_trades="sum",
).reset_index()
display(zero_trade_summary)

coverage_checks = [
    check_row("walk-forward produced windows for both D13 strategies", set(D13_WINDOW_DF["strategy_name"]) == set(D13_NEW_STRATEGIES)),
    check_row("each strategy has same window count as run result", all(
        len(D13_WINDOW_DF[D13_WINDOW_DF["strategy_name"] == name]) == len(result.windows)
        for name, result in D13_WF_RESULTS.items()
    )),
    check_row("every test window has more bars than strategy warmup", (D13_WINDOW_DF["num_test_bars"] > D13_WINDOW_DF["warmup_bars"]).all(), D13_WINDOW_DF.loc[D13_WINDOW_DF["num_test_bars"] <= D13_WINDOW_DF["warmup_bars"]].to_dict(orient="records")),
    check_row("window metrics are finite", D13_WINDOW_DF[["sharpe_ratio", "total_return", "max_drawdown"]].map(finite_number).all().all()),
    check_row("each D13 strategy has at least one walk-forward trade overall", (zero_trade_summary["total_trades"] > 0).all(), zero_trade_summary.to_dict(orient="records")),
]
display_check_table(coverage_checks, "D13 walk-forward coverage checks")
D13_ACCEPTANCE["walk_forward_coverage"] = True


2026-04-16T06:37:04Z [INFO] Walk-forward: 20 windows, volatility_breakout strategy, train=12m test=3m step=3m
2026-04-16T06:37:04Z [INFO]   Window 1/20: train=[2020-01-01, 2020-12-31] test=[2021-01-01, 2021-03-31]
2026-04-16T06:37:04Z [INFO] Starting backtest: strategy=volatility_breakout, range=[2020-01-01, 2021-03-31], run_id=b5130730
2026-04-16T06:37:04Z [INFO] Loading parquet feed: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/raw/btcusdt_1h.parquet
2026-04-16T06:37:04Z [INFO]   Loaded 10925 bars: 2020-01-01 00:00:00 to 2021-03-31 23:00:00
2026-04-16T06:37:04Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T06:37:04Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%
2026-04-16T06:37:05Z [INFO] Metrics: return=0.5991 sharpe=1.111 maxDD=0.2407 trades=190 win_rate=0.34 PF=1.25
2026-04-16T06:37:05Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline

,strategy_name,summary_run_id,run_id,test_start,test_end,num_test_bars,warmup_bars,total_trades,sharpe_ratio,total_return,max_drawdown
0,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,b5130730-92b8-492d-9fc9-20001575aa55,2021-01-01T00:00:00Z,2021-03-31T23:00:00Z,2158,24,39,0.566783,0.599122,0.209236
1,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,ec95ce63-26e1-4862-9291-49a9c35790df,2021-04-01T00:00:00Z,2021-06-30T23:00:00Z,2179,24,29,0.786833,0.814387,0.120207
2,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,f7a5b94e-70ba-49a7-9fc1-41322b4bd6f6,2021-07-01T00:00:00Z,2021-09-30T23:00:00Z,2202,24,35,1.506751,0.826328,0.093661
3,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,2a5fd704-7e39-476d-a073-ba35a1754037,2021-10-01T00:00:00Z,2021-12-31T23:00:00Z,2208,24,41,-1.558277,0.813108,0.228445
4,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,954cb7ad-c1d9-4586-bd3f-974199f911a8,2022-01-01T00:00:00Z,2022-03-31T23:00:00Z,2160,24,37,-0.789986,0.002446,0.175058
5,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,df1314b9-2988-4321-b22d-e359dee8a6e3,2022-04-01T00:00:00Z,2022-06-30T23:00:00Z,2184,24,31,-1.827046,-0.180236,0.171112
6,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,977a40a0-c72b-466b-a772-d65c0f0ebde1,2022-07-01T00:00:00Z,2022-09-30T23:00:00Z,2208,24,41,-2.949759,-0.421266,0.252971
7,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,7e9ba1ca-b9fd-4891-97f4-33b129d72564,2022-10-01T00:00:00Z,2022-12-31T23:00:00Z,2208,24,29,0.508498,-0.498997,0.097052
8,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,0353db0a-e4cb-41ad-a40f-6c01fca5fd63,2023-01-01T00:00:00Z,2023-03-31T23:00:00Z,2159,24,37,3.649544,-0.145961,0.165550
9,volatility_breakout,c0e2e175-31bb-4a0d-9a66-d80cb391678b,a5569c0a-0d32-4ea9-9412-6a87129f23dc,2023-04-01T00:00:00Z,2023-06-30T23:00:00Z,2184,24,41,0.091853,-0.084837,0.176103


,strategy_name,windows,zero_trade_windows,total_trades
0,mean_reversion,20,0,417
1,volatility_breakout,20,0,734


D13 walk-forward coverage checks


,check,status,detail
0,walk-forward produced windows for both D13 strategies,PASS,
1,each strategy has same window count as run result,PASS,
2,every test window has more bars than strategy warmup,PASS,[]
3,window metrics are finite,PASS,
4,each D13 strategy has at least one walk-forward trade overall,PASS,"[{'strategy_name': 'mean_reversion', 'windows': 20, 'zero_trade_windows': 0, 'total_trades': 417}, {'strategy_name': 'volatility_breakout', 'windows': 20, '..."


## S. Walk-Forward Artifact Consistency

Reuses the D11 artifact isolation checks for the two new strategies: persisted test-window CSVs must match registry `total_trades`, and entries/exits must stay inside each window.


In [26]:
artifact_rows = []
leak_rows = []

for _, row in D13_WINDOW_DF.iterrows():
    trade_path, trades = load_trade_csv(row["run_id"])
    registry_total = int(row["total_trades"])
    test_start = as_utc_timestamp(row["test_start"])
    test_end = as_utc_timestamp(row["test_end"])

    csv_expected = registry_total > 0
    row_summary = {
        "strategy_name": row["strategy_name"],
        "run_id": row["run_id"],
        "registry_total_trades": registry_total,
        "trade_csv_exists": trade_path.exists(),
        "trade_csv_required": csv_expected,
        "csv_rows": len(trades),
        "row_count_matches_registry": (len(trades) == registry_total) if trade_path.exists() else (registry_total == 0),
        "all_entries_in_test": True,
        "all_exits_in_test": True,
    }

    if registry_total > 0 and not trade_path.exists():
        row_summary["all_entries_in_test"] = False
        row_summary["all_exits_in_test"] = False
        leak_rows.append({"run_id": row["run_id"], "issue": "missing required trade CSV"})
    elif len(trades) > 0:
        entry_ok = (trades["entry_time_utc"] >= test_start) & (trades["entry_time_utc"] <= test_end)
        exit_ok = (trades["exit_time_utc"] >= test_start) & (trades["exit_time_utc"] <= test_end)
        row_summary["all_entries_in_test"] = bool(entry_ok.all())
        row_summary["all_exits_in_test"] = bool(exit_ok.all())
        leaked = trades[~(entry_ok & exit_ok)].copy()
        if len(leaked) > 0:
            leaked.insert(0, "strategy_name", row["strategy_name"])
            leaked.insert(1, "run_id", row["run_id"])
            leaked.insert(2, "test_start", test_start)
            leaked.insert(3, "test_end", test_end)
            leak_rows.extend(leaked[[
                "strategy_name", "run_id", "test_start", "test_end", "trade_id",
                "entry_signal_time_utc", "entry_time_utc",
                "exit_signal_time_utc", "exit_time_utc",
            ]].to_dict(orient="records"))

    artifact_rows.append(row_summary)

D13_ARTIFACT_DF = pd.DataFrame(artifact_rows)
display(D13_ARTIFACT_DF)

D13_LEAK_DF = pd.DataFrame(leak_rows)
if len(D13_LEAK_DF) > 0:
    display(D13_LEAK_DF.head(30))

artifact_checks = [
    check_row("trade CSV exists whenever registry total_trades > 0", bool((D13_ARTIFACT_DF["trade_csv_exists"] | ~D13_ARTIFACT_DF["trade_csv_required"]).all())),
    check_row("CSV row counts match registry total_trades", bool(D13_ARTIFACT_DF["row_count_matches_registry"].all()), D13_ARTIFACT_DF.loc[~D13_ARTIFACT_DF["row_count_matches_registry"]].to_dict(orient="records")),
    check_row("all trade entries are inside their test window", bool(D13_ARTIFACT_DF["all_entries_in_test"].all()), f"leaked_rows={len(D13_LEAK_DF)}"),
    check_row("all trade exits are inside their test window", bool(D13_ARTIFACT_DF["all_exits_in_test"].all()), f"leaked_rows={len(D13_LEAK_DF)}"),
]
display_check_table(artifact_checks, "D13 walk-forward artifact consistency checks")
D13_ACCEPTANCE["artifact_consistency"] = True


,strategy_name,run_id,registry_total_trades,trade_csv_exists,trade_csv_required,csv_rows,row_count_matches_registry,all_entries_in_test,all_exits_in_test
0,volatility_breakout,b5130730-92b8-492d-9fc9-20001575aa55,39,True,True,39,True,True,True
1,volatility_breakout,ec95ce63-26e1-4862-9291-49a9c35790df,29,True,True,29,True,True,True
2,volatility_breakout,f7a5b94e-70ba-49a7-9fc1-41322b4bd6f6,35,True,True,35,True,True,True
3,volatility_breakout,2a5fd704-7e39-476d-a073-ba35a1754037,41,True,True,41,True,True,True
4,volatility_breakout,954cb7ad-c1d9-4586-bd3f-974199f911a8,37,True,True,37,True,True,True
5,volatility_breakout,df1314b9-2988-4321-b22d-e359dee8a6e3,31,True,True,31,True,True,True
6,volatility_breakout,977a40a0-c72b-466b-a772-d65c0f0ebde1,41,True,True,41,True,True,True
7,volatility_breakout,7e9ba1ca-b9fd-4891-97f4-33b129d72564,29,True,True,29,True,True,True
8,volatility_breakout,0353db0a-e4cb-41ad-a40f-6c01fca5fd63,37,True,True,37,True,True,True
9,volatility_breakout,a5569c0a-0d32-4ea9-9412-6a87129f23dc,41,True,True,41,True,True,True


D13 walk-forward artifact consistency checks


,check,status,detail
0,trade CSV exists whenever registry total_trades > 0,PASS,
1,CSV row counts match registry total_trades,PASS,[]
2,all trade entries are inside their test window,PASS,leaked_rows=0
3,all trade exits are inside their test window,PASS,leaked_rows=0


## T. One Manual Trade Audit Per New Strategy

Audits one real trade from each D13 single-run artifact against raw parquet: next-bar-open fill, non-zero volume, and 7 bps per-side commission.


In [27]:
raw_audit = raw_for_d13.set_index("open_time_utc").sort_index()
audit_rows = []

def raw_bar_at(ts):
    ts = as_utc_timestamp(ts)
    assert ts in raw_audit.index, f"Missing raw bar for {ts}"
    return raw_audit.loc[ts]

for strategy_name, result in D13_SINGLE_RESULTS.items():
    trade_path, trades = load_trade_csv(result.run_id)
    assert len(trades) > 0, f"No trade available to audit for {strategy_name}"
    trade = trades.iloc[0]

    entry_signal = as_utc_timestamp(trade["entry_signal_time_utc"])
    entry_time = as_utc_timestamp(trade["entry_time_utc"])
    exit_signal = as_utc_timestamp(trade["exit_signal_time_utc"])
    exit_time = as_utc_timestamp(trade["exit_time_utc"])
    entry_bar = raw_bar_at(entry_time)
    exit_bar = raw_bar_at(exit_time)

    size = abs(float(trade["size"]))
    expected_entry_comm = float(trade["entry_price"]) * size * D13_FEE_RATE
    expected_exit_comm = float(trade["exit_price"]) * size * D13_FEE_RATE

    audit_rows.append({
        "strategy_name": strategy_name,
        "run_id": result.run_id,
        "trade_id": int(trade["trade_id"]),
        "entry_signal_time_utc": entry_signal,
        "entry_time_utc": entry_time,
        "exit_signal_time_utc": exit_signal,
        "exit_time_utc": exit_time,
        "entry_signal_before_fill": entry_signal < entry_time,
        "entry_is_next_bar": entry_time - entry_signal == pd.Timedelta(hours=1),
        "entry_price_matches_open": np.isclose(float(trade["entry_price"]), float(entry_bar["open"]), rtol=0, atol=1e-12),
        "entry_volume_positive": float(entry_bar["volume"]) > 0,
        "exit_signal_before_fill": exit_signal < exit_time,
        "exit_is_next_bar": exit_time - exit_signal == pd.Timedelta(hours=1),
        "exit_price_matches_open": np.isclose(float(trade["exit_price"]), float(exit_bar["open"]), rtol=0, atol=1e-12),
        "exit_volume_positive": float(exit_bar["volume"]) > 0,
        "entry_commission_7bps": np.isclose(float(trade["entry_commission"]), expected_entry_comm, rtol=0, atol=1e-10),
        "exit_commission_7bps": np.isclose(float(trade["exit_commission"]), expected_exit_comm, rtol=0, atol=1e-10),
        "total_commission_matches": np.isclose(float(trade["total_commission"]), float(trade["entry_commission"]) + float(trade["exit_commission"]), rtol=0, atol=1e-10),
    })

D13_AUDIT_DF = pd.DataFrame(audit_rows)
display(D13_AUDIT_DF)

audit_bool_cols = [
    "entry_signal_before_fill", "entry_is_next_bar", "entry_price_matches_open", "entry_volume_positive",
    "exit_signal_before_fill", "exit_is_next_bar", "exit_price_matches_open", "exit_volume_positive",
    "entry_commission_7bps", "exit_commission_7bps", "total_commission_matches",
]
audit_checks = [
    check_row("one trade audited for each D13 strategy", set(D13_AUDIT_DF["strategy_name"]) == set(D13_NEW_STRATEGIES)),
    check_row("all manual trade audit checks pass", D13_AUDIT_DF[audit_bool_cols].all().all(), D13_AUDIT_DF.loc[~D13_AUDIT_DF[audit_bool_cols].all(axis=1)].to_dict(orient="records")),
]
display_check_table(audit_checks, "D13 manual trade audit checks")
D13_ACCEPTANCE["manual_trade_audit"] = True


,strategy_name,run_id,trade_id,entry_signal_time_utc,entry_time_utc,exit_signal_time_utc,exit_time_utc,entry_signal_before_fill,entry_is_next_bar,entry_price_matches_open,entry_volume_positive,exit_signal_before_fill,exit_is_next_bar,exit_price_matches_open,exit_volume_positive,entry_commission_7bps,exit_commission_7bps,total_commission_matches
0,volatility_breakout,4f51640e-3cde-4f23-beb2-2a4cc2770cef,1,2024-01-01 23:00:00+00:00,2024-01-02 00:00:00+00:00,2024-01-02 19:00:00+00:00,2024-01-02 20:00:00+00:00,True,True,True,True,True,True,True,True,True,True,True
1,mean_reversion,893c9858-b3b5-4f41-b63c-f61513091f96,1,2024-01-03 13:00:00+00:00,2024-01-03 14:00:00+00:00,2024-01-04 15:00:00+00:00,2024-01-04 16:00:00+00:00,True,True,True,True,True,True,True,True,True,True,True


D13 manual trade audit checks


,check,status,detail
0,one trade audited for each D13 strategy,PASS,
1,all manual trade audit checks pass,PASS,[]


## U. Four-Baseline Sanity Table

Runs any missing existing baselines on the same fixed range and compares all four baseline strategies. This is not a profitability target; it is a smoke test for suspiciously inflated Sharpe, explosive trade count, or non-finite metrics.


In [28]:
D13_BASELINE_RESULTS = dict(D13_SINGLE_RESULTS)
for strategy_name, strategy_cls in D13_ALL_BASELINES.items():
    if strategy_name in D13_BASELINE_RESULTS:
        continue
    D13_BASELINE_RESULTS[strategy_name] = run_backtest(
        strategy_cls=strategy_cls,
        start_date=D13_SINGLE_START,
        end_date=D13_SINGLE_END,
        parquet_path=PARQUET_PATH,
        cash=D13_CASH,
        write_registry=True,
    )

baseline_rows = []
for strategy_name, result in D13_BASELINE_RESULTS.items():
    baseline_rows.append({
        "strategy_name": strategy_name,
        "run_id": result.run_id,
        "warmup_bars": result.warmup_bars,
        "trades": len(result.trades),
        "sharpe_ratio": result.metrics["sharpe_ratio"],
        "total_return": result.metrics["total_return"],
        "max_drawdown": result.metrics["max_drawdown"],
    })

D13_BASELINE_DF = pd.DataFrame(baseline_rows).sort_values("strategy_name").reset_index(drop=True)
display(D13_BASELINE_DF)

sanity_checks = [
    check_row("all four baselines represented", set(D13_BASELINE_DF["strategy_name"]) == set(D13_ALL_BASELINES)),
    check_row("all baselines produce at least one trade", (D13_BASELINE_DF["trades"] > 0).all(), D13_BASELINE_DF[["strategy_name", "trades"]].to_dict(orient="records")),
    check_row("trade counts are not explosive", (D13_BASELINE_DF["trades"] < 5_000).all(), D13_BASELINE_DF[["strategy_name", "trades"]].to_dict(orient="records")),
    check_row("metrics are finite", D13_BASELINE_DF[["sharpe_ratio", "total_return", "max_drawdown"]].map(finite_number).all().all()),
    check_row("Sharpe values are not obviously engine-bug inflated", (D13_BASELINE_DF["sharpe_ratio"].abs() < 5.0).all(), D13_BASELINE_DF[["strategy_name", "sharpe_ratio"]].to_dict(orient="records")),
]
display_check_table(sanity_checks, "D13 four-baseline sanity checks")
D13_ACCEPTANCE["sanity"] = True


2026-04-16T06:37:56Z [INFO] Starting backtest: strategy=sma_crossover, range=[2024-01-01, 2024-06-30], run_id=d8ee80b1
2026-04-16T06:37:56Z [INFO] Loading parquet feed: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/raw/btcusdt_1h.parquet
2026-04-16T06:37:56Z [INFO]   Loaded 4368 bars: 2024-01-01 00:00:00 to 2024-06-30 23:00:00
2026-04-16T06:37:56Z [INFO] Cost model: effective_7bps_per_side (fee=4.0bps + slip=3.0bps = 7.0bps per side)
2026-04-16T06:37:56Z [INFO] Cerebro configured: cash=10000.00, coc=False, coo=False, effective_7bps_per_side, sizer=99%
2026-04-16T06:37:56Z [INFO] Metrics: return=-0.0067 sharpe=0.141 maxDD=0.3178 trades=56 win_rate=0.38 PF=0.96
2026-04-16T06:37:56Z [INFO] Trade log saved: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline/data/results/trades_d8ee80b1-f026-42a5-baf5-8e9ff64b93ab.csv (56 trades)
2026-04-16T06:37:56Z [INFO] Ensured 'runs' table exists with Phase 1A schema
2026-04-16T06:37:56Z [INFO] Inserted run d8ee80b1-f026-42a5-baf5-8e9ff

,strategy_name,run_id,warmup_bars,trades,sharpe_ratio,total_return,max_drawdown
0,mean_reversion,893c9858-b3b5-4f41-b63c-f61513091f96,48,46,1.365857,0.203350,0.153280
1,momentum,59ea66a5-2174-4a26-a8b9-ee44c444a19d,24,61,-0.685641,-0.114175,0.296563
2,sma_crossover,d8ee80b1-f026-42a5-baf5-8e9ff64b93ab,50,56,0.141497,-0.006726,0.317832
3,volatility_breakout,4f51640e-3cde-4f23-beb2-2a4cc2770cef,24,80,0.825532,0.101031,0.147404


D13 four-baseline sanity checks


,check,status,detail
0,all four baselines represented,PASS,
1,all baselines produce at least one trade,PASS,"[{'strategy_name': 'mean_reversion', 'trades': 46}, {'strategy_name': 'momentum', 'trades': 61}, {'strategy_name': 'sma_crossover', 'trades': 56}, {'strateg..."
2,trade counts are not explosive,PASS,"[{'strategy_name': 'mean_reversion', 'trades': 46}, {'strategy_name': 'momentum', 'trades': 61}, {'strategy_name': 'sma_crossover', 'trades': 56}, {'strateg..."
3,metrics are finite,PASS,
4,Sharpe values are not obviously engine-bug inflated,PASS,"[{'strategy_name': 'mean_reversion', 'sharpe_ratio': 1.3658570447389082}, {'strategy_name': 'momentum', 'sharpe_ratio': -0.6856414219105897}, {'strategy_nam..."


## V. D13 Final Acceptance Summary

D13 passes when the new baselines are discoverable, complete single-run and walk-forward checks, preserve artifact isolation, pass one real trade audit each, and look sane beside the existing baselines.


In [29]:
d13_rows = [
    check_row("strategy discovery", D13_ACCEPTANCE.get("discovery", False)),
    check_row("single-run spot checks", D13_ACCEPTANCE.get("single_run", False)),
    check_row("walk-forward window coverage", D13_ACCEPTANCE.get("walk_forward_coverage", False)),
    check_row("artifact consistency", D13_ACCEPTANCE.get("artifact_consistency", False)),
    check_row("manual trade audit", D13_ACCEPTANCE.get("manual_trade_audit", False)),
    check_row("four-baseline sanity", D13_ACCEPTANCE.get("sanity", False)),
]
display_check_table(d13_rows, "D13 final acceptance summary")
print("D13 ACCEPTED: additional baselines preserve Phase 1B discovery, registry, artifact, window, and execution semantics.")


D13 final acceptance summary


,check,status,detail
0,strategy discovery,PASS,
1,single-run spot checks,PASS,
2,walk-forward window coverage,PASS,
3,artifact consistency,PASS,
4,manual trade audit,PASS,
5,four-baseline sanity,PASS,


D13 ACCEPTED: additional baselines preserve Phase 1B discovery, registry, artifact, window, and execution semantics.
